# RFDC V4 -- Clean RX-only detection diagnostic + Ethernet overlap

Production-path diagnostic: removes the obsolete one-row TX-B qualification from Run All, keeps the validated row/refill/Ethernet-overlap logic, makes the final detection row RX-only, and records TX gate status immediately around the RX firing time.


## Cell 1 -- Overlay + imports (only this)

In [1]:
# ================= OVERLAY + IMPORTS (run this once) =================
import time
import numpy as np
import xrfclk
import xrfdc
from pynq import Overlay, allocate

BITFILE = "./final.bit"   # confirm this is the 256-row Pulse_Sequencer build
base = Overlay(BITFILE)
print("Overlay loaded:", BITFILE)


Overlay loaded: ./final.bit


## Cell 2 -- User configuration

In [2]:
# ================= USER CONFIG =================
AXIS_BEAT_HZ = 15.36e6
SAMPLES_PER_BEAT = 8
SEQ_CLK_HZ = 99_999_985.0
FS_HZ = 122.88e6
AMPLITUDE = 32760

DAC_A_NCO_MHZ = 20.0
DAC_B_NCO_MHZ = 20.0

DMA_MAX_BYTES = (1 << 26) - 1
CAP_MAX_S = 0.260
TX_MAX_S = 0.136

LOOP_COUNT = 5           # keep small for a bench run; bump once margins look safe
DEBUG_VERBOSE = True

EXPECTED_IP_PATHS = {
    "sequencer": "radio/AXI_Pulse_Sequencer_0",
    "tx_gate_b": "radio/AXI_TX_Multi_Gate_0",
    "tx_gate_a": "radio/AXI_TX_Multi_Gate_1",
    "cap_gate_b": "radio/receiver/channel_20/AXI_Capture_Gate_0",
    "cap_gate_a": "radio/receiver/channel_21/AXI_Capture_Gate_0",
    "rx_dma_b": "radio/receiver/channel_20/axi_dma_real",
    "rx_dma_a": "radio/receiver/channel_21/axi_dma_real",
    "tx_dma_b": "radio/axi_dma_dac_0",
    "tx_dma_a": "radio/axi_dma_dac_1",
}
print("Config loaded.")

# ================= ETHERNET CONFIG =================
PC_HOST = "192.168.3.139"   # PC receiver IP; edit if needed
PC_PORT = 5001
ETH_TIMEOUT_S = 30.0
FAIL_FAST = True
# Conservative planning model only; runtime never assumes a past transfer predicts the next one.
ETHERNET_MODEL_MBPS = 200.0
ETHERNET_MODEL_MARGIN_S = 10e-3
RX_ARM_GUARD_S = 5e-3
# Policy: never delay the Pulse Sequencer for Ethernet. If the previous RX buffer is still
# owned by Ethernet at the RX-arm deadline, mark this run's capture dropped.


Config loaded.


## Cell 3 -- IP resolution

In [3]:
# ================= IP RESOLUTION =================
def get_by_path(root, path):
    obj = root
    for part in path.split('/'):
        obj = getattr(obj, part)
    return obj

missing = [p for p in EXPECTED_IP_PATHS.values() if p not in base.ip_dict]
if missing:
    print("Missing expected HWH paths:")
    for p in missing:
        print("  ", p)
    raise KeyError("The loaded .hwh does not match the validated A/B topology.")

seq   = get_by_path(base, EXPECTED_IP_PATHS['sequencer'])
tx_b  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_b'])
tx_a  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_a'])
cap_b = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_b'])
cap_a = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_a'])
dma_rb = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_b'])
dma_ra = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_a'])
dma_tb = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_b'])
dma_ta = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_a'])

rfdc = get_by_path(base, 'radio/rfdc')
dac_b = rfdc.dac_tiles[0].blocks[0]
dac_a = rfdc.dac_tiles[2].blocks[0]
adc_b = rfdc.adc_tiles[2].blocks[0]
adc_a = rfdc.adc_tiles[2].blocks[1]
print('IP resolved.')


IP resolved.


## Cell 4 -- Register map + unit-conversion helpers

In [4]:
# ================= REGISTER MAP =================
SEQ_ROW_SEL=0x00; SEQ_MASK=0x04; SEQ_GAP=0x08
SEQ_DUR0=0x0C; SEQ_DUR1=0x10; SEQ_DUR2=0x14; SEQ_DUR3=0x18
SEQ_COMMIT=0x1C; SEQ_TABLE_LEN=0x20; SEQ_ENABLE=0x24
SEQ_RESET=0x28; SEQ_ROW_PTR=0x2C; SEQ_CDC_OVERRUN=0x30
SEQ_TTL_ARM=0x34; SEQ_TTL_MODE=0x38; SEQ_TTL_STATUS=0x3C
SEQ_TTL_STATUS_CLEAR=0x40; SEQ_TTL_EDGE_COUNT=0x44
TTL_ARMED=1<<0; TTL_RUNNING=1<<1; TTL_RUN_DONE=1<<2

TXM_SEG_SEL=0x00; TXM_ACTIVE_LEN=0x04; TXM_GAP_LEN=0x08; TXM_COMMIT=0x0C
TXM_NUM_SEGMENTS=0x10; TXM_SW_START=0x14; TXM_SEG_PTR=0x18; TXM_STATUS=0x1C
# NOTE: TXM_SW_START (0x14) and TXM_SEG_PTR (0x18) exist in AXI_TX_Multi_Gate.vhd's
# register map but were missing here. SW_START independently forces the gate open
# using INTERNAL-table mode, regardless of the Pulse_Sequencer/hw_start path -- this
# is the only way to open a gate for a bench/debug test without a committed sequencer
# program. TX_OVERRUN here is actually STATUS bit1 = CONTROL_OVERRUN per the VHDL
# comment (COMMIT/SW_START issued too close together), not a data overrun flag; kept
# the existing name so the rest of the notebook does not need touching.
TX_BUSY=1<<0; TX_OVERRUN=1<<1

CAP_LENGTH=0x00; CAP_STATUS=0x08; CAP_CLEAR=0x0C
CAP_BUSY=1<<0; CAP_OVERFLOW=1<<1

DMASR_HALTED=1<<0; DMASR_IDLE=1<<1; DMASR_ERR_MASK=(1<<4)|(1<<5)|(1<<6)

LANE_TX_B, LANE_TX_A, LANE_RX_B, LANE_RX_A = 0, 1, 2, 3
LANE_BIT = {"TX_B": LANE_TX_B, "TX_A": LANE_TX_A, "RX_B": LANE_RX_B, "RX_A": LANE_RX_A}
TX_LANES = ("TX_A", "TX_B")
RX_LANES = ("RX_A", "RX_B")

def beats_for(seconds):
    return max(1, int(round(seconds * AXIS_BEAT_HZ)))

def samples_for(beats):
    return int(beats) * SAMPLES_PER_BEAT

def seq_cycles_for(seconds):
    return max(1, int(round(seconds * SEQ_CLK_HZ)))

def pack_iq(i, q):
    if len(i) != len(q):
        raise ValueError('I/Q length mismatch')
    out = np.empty(2*len(i), dtype=np.int16)
    out[0::2] = i
    out[1::2] = q
    return out
print('Register map ready.')


Register map ready.


## Cell 5 -- Waveform primitives

Unchanged from the row-program design: `chirp()` (sine is `f0==f1`), `zeros()`,
`envelope()`, `waveform(*parts)`, `quantize_iq()` (final step only), `chirp_train()`.

In [5]:
# ================= WAVEFORM PRIMITIVES =================
def chirp(f0_mhz, f1_mhz, duration_s, nco_mhz):
    n = samples_for(beats_for(duration_s))
    t = np.arange(n, dtype=np.float64) / FS_HZ
    f0 = (f0_mhz - nco_mhz) * 1e6
    f1 = (f1_mhz - nco_mhz) * 1e6
    k = (f1 - f0) / duration_s if duration_s > 0 else 0.0
    ph = 2*np.pi*(f0*t + 0.5*k*t*t)
    return np.exp(1j*ph).astype(np.complex128)

def zeros(duration_s):
    n = samples_for(beats_for(duration_s))
    return np.zeros(n, dtype=np.complex128)

def envelope(arr, kind="gaussian", **params):
    n = len(arr)
    if kind == "gaussian":
        sigma = params.get("sigma", 0.25) * n
        x = np.arange(n) - (n-1)/2.0
        win = np.exp(-0.5*(x/sigma)**2)
    else:
        raise ValueError(f"unknown envelope kind: {kind}")
    return arr * win

def waveform(*parts):
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.complex128)

def quantize_iq(arr):
    i = np.round(AMPLITUDE * arr.real).astype(np.int16)
    q = np.round(AMPLITUDE * arr.imag).astype(np.int16)
    return pack_iq(i, q)

def chirp_train(freqs_mhz, pulse_s, gap_s, nco_mhz):
    parts = []
    for idx, f in enumerate(freqs_mhz):
        parts.append(chirp(f, f, pulse_s, nco_mhz))
        if idx < len(freqs_mhz) - 1:
            parts.append(zeros(gap_s))
    return waveform(*parts)

print('Waveform primitives ready.')


Waveform primitives ready.


## Cell 6 -- Row helpers: `repeat()` and `print_program()`

In [6]:
# ================= ROW HELPERS =================
def repeat(rows, n):
    return list(rows) * n

def _row_lanes(row):
    lanes = []
    for lane in TX_LANES:
        if lane in row.get("tx", {}):
            lanes.append(lane)
    for lane in RX_LANES:
        if lane in row.get("rx", {}):
            lanes.append(lane)
    return lanes

def print_program(rows):
    print(
        f"{'row':>4}  "
        f"{'gap_after_s':>13}  "
        f"{'lanes (dur_s)':<50} "
        f"{'refill'}"
    )

    for idx, row in enumerate(rows):

        parts = []

        for lane in _row_lanes(row):

            if lane in TX_LANES:

                arr = row["tx"][lane]

                dur_s = (
                    len(arr)
                    / SAMPLES_PER_BEAT
                    / AXIS_BEAT_HZ
                )

                parts.append(
                    f"{lane}:{dur_s*1e3:.4f}ms"
                    f"({len(arr)}smp)"
                )

            else:

                dur_s = row["rx"][lane]

                parts.append(
                    f"{lane}:{dur_s*1e3:.4f}ms"
                )

        refill = (
            ",".join(row.get("refill", []))
            or "-"
        )

        print(
            f"{idx:>4}  "
            f"{row['gap_after_s']*1e3:>12.4f}ms  "
            f"{', '.join(parts):<50} "
            f"{refill}"
        )

print('repeat() and print_program() ready.')


repeat() and print_program() ready.


## Cell 7 -- The program (same sequence as the design that hung, unchanged on purpose)

Kept identical to `RFDC_V4_RowProgram_Design_FIXED_2_.ipynb`'s state-prep / spectroscopy /
detection example so this bench is testing the same timing that failed, not an easier case.

In [7]:
# ================= PROGRAM DEFINITION =================
#
# USER-FACING TIMING SEMANTICS
#
# gap_after_s means:
#
#     THIS row fires
#         |
#         | gap_after_s
#         v
#     NEXT row fires
#
# The hardware Pulse Sequencer uses the opposite representation:
# SEQ_GAP belongs BEFORE the row being fired.
#
# Cell 8 performs that translation automatically.


ROW_MARGIN_S = 5e-6

# Small safe delay from accepted TTL trigger to user row 0.
INITIAL_GAP_S = 8 / SEQ_CLK_HZ


def gap_after(*durations_s):
    """
    Convenience helper:
    next row fires after the longest supplied duration
    plus ROW_MARGIN_S.
    """
    return max(durations_s) + ROW_MARGIN_S


# ============================================================
# STATE PREPARATION WAVEFORMS
# ============================================================

state_prep_dacA = waveform(
    chirp(
        30,
        30,
        1e-3,
        DAC_A_NCO_MHZ,
    ),
    chirp(
        20,
        30,
        1e-3,
        DAC_A_NCO_MHZ,
    ),
)


state_prep_dacB = chirp(
    20,
    30,
    20e-3,
    DAC_B_NCO_MHZ,
)


STATE_PREP_REPS = 20


# ============================================================
# STATE PREPARATION ROWS
#
# Intended physical sequence:
#
#   TX_A starts
#       1 ms sine
#       1 ms chirp
#
#   2.005 ms after TX_A START:
#       TX_B starts
#       20 ms chirp
#
#   20.005 ms after TX_B START:
#       next TX_A starts
#
# repeat 20 times.
#
# Thus there should be essentially NO A/B overlap other than
# the intentional 5-us safety-margin convention.
# ============================================================

state_prep_rows = []


for rep in range(STATE_PREP_REPS):

    # --------------------------------------------------------
    # TX_A row
    #
    # TX_B fires 2.005 ms after this row fires.
    #
    # For reps 1..19, this same ~2-ms interval is also where
    # software loads the NEXT TX_B DMA chunk.
    #
    # TX_B chunk 0 is armed before TTL_ARM, so rep 0 does not
    # need a refill.
    # --------------------------------------------------------

    row_a = {
        "tx": {
            "TX_A": state_prep_dacA,
        },

        "gap_after_s": gap_after(
            2e-3
        ),
    }


    if rep > 0:

        row_a["refill"] = [
            "TX_B"
        ]


    state_prep_rows.append(
        row_a
    )


    # --------------------------------------------------------
    # TX_B row
    #
    # The NEXT TX_A row normally fires 20.005 ms after this
    # TX_B row starts.
    #
    # On the FINAL repetition, the next row is instead the
    # empty spectroscopy marker. That is also correct:
    #
    #     final TX_B starts
    #       |
    #       | 20.005 ms
    #       v
    #     spectroscopy marker fires
    #
    # so final TX_B is finished before spectroscopy begins.
    # --------------------------------------------------------

    row_b = {
        "tx": {
            "TX_B": state_prep_dacB,
        },

        "gap_after_s": gap_after(
            20e-3
        ),
    }


    state_prep_rows.append(
        row_b
    )


# ============================================================
# SPECTROSCOPY
#
# This row itself has no RFSoC TX/RX action.
#
# When it fires, the final TX_B pulse has already completed.
#
# Detection fires 5.005 ms after this row.
# ============================================================

SPECTROSCOPY_S = 5e-3


spectroscopy_rows = [
    {
        "tx": {},
        "rx": {},

        "gap_after_s": gap_after(
            SPECTROSCOPY_S
        ),
    },
]


# ============================================================
# PROBE DEFINITION
#
# Keep this generated because CAPTURE_S is still based on the
# normal five-pulse detection waveform.
#
# For THIS diagnostic, however, neither DAC transmits during
# detection. Once RX_B is confirmed quiet, TX_A probe can be
# restored.
# ============================================================

probe = waveform(
    chirp_train(
        [20, 21, 22, 23],
        pulse_s=10e-6,
        gap_s=1e-6,
        nco_mhz=DAC_A_NCO_MHZ,
    ),

    zeros(1e-6),

    envelope(
        chirp(
            24,
            24,
            10e-6,
            DAC_A_NCO_MHZ,
        ),
        kind="gaussian",
        sigma=1/6,
    ),
)

PROBE_S = (
    len(probe)
    / SAMPLES_PER_BEAT
    / AXIS_BEAT_HZ
)

RX_TAIL_S = 5e-6

CAPTURE_S = (
    PROBE_S
    + RX_TAIL_S
)


# ============================================================
# DETECTION
#
# DIAGNOSTIC VERSION:
#
#   TX_A OFF
#   TX_B OFF
#   RX_A ON
#   RX_B ON
#
# Both TX gates therefore should be CLOSED throughout this
# capture.
#
# gap_after_s on the final row is used only for total timing
# bookkeeping; there is no subsequent experimental row.
# ============================================================

detection_rows = [
    {
        "tx": {
            "TX_A": probe,
            "TX_B": probe,
        },

        "rx": {
            "RX_A": CAPTURE_S,
            "RX_B": CAPTURE_S,
        },

        "gap_after_s": CAPTURE_S,
    },
]


# ============================================================
# FINAL USER PROGRAM
# ============================================================

rows = (
    state_prep_rows
    + spectroscopy_rows
    + detection_rows
)


print()
print(
    f"User program contains {len(rows)} rows."
)

print()
print_program(rows)



User program contains 42 rows.

 row    gap_after_s  lanes (dur_s)                                      refill
   0        2.0050ms  TX_A:2.0000ms(245760smp)                           -
   1       20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   2        2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   3       20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   4        2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   5       20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   6        2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   7       20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   8        2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   9       20.0050ms  TX_B:20.0000ms(2457600smp)                         -
  10        2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
  11       20.0050ms  TX_B:20.0000ms(2457600smp)

## Cell 8 -- Compile: derive mask/dur, ONE buffer per lane, refill deadlines

Same mask/DUR derivation as before. New: `row_start_s[idx]` models the wall-clock time (from
`TTL_ARM`) each row *starts*, purely from the committed `gap_cycles` -- this is what lets the
run loop print a real number for "how much margin did this refill actually have" instead of
just hoping a 2 ms window was enough.

In [8]:
# ================= COMPILE PROGRAM =================
#
# USER SEMANTICS:
#
#     row N fires
#         |
#         | row[N]["gap_after_s"]
#         v
#     row N+1 fires
#
# HARDWARE SEMANTICS:
#
#     wait SEQ_GAP[N]
#     row N fires
#
# Therefore:
#
#     HW_GAP[0] = INITIAL_GAP_S
#     HW_GAP[N] = USER gap_after_s[N-1], N > 0
#
# This compiler:
#
#   - derives lane masks
#   - derives TX durations from waveform lengths
#   - derives RX durations
#   - concatenates per-lane TX streams
#   - creates DMA chunk boundaries from refill markers
#   - calculates exact hardware row-fire times
#   - calculates refill windows
#   - allows intentional simultaneous TX lanes in one row
#   - rejects unintended TX overlap BETWEEN different rows
#   - verifies all TX activity BEFORE first RX has finished
#   - allows intentional TX + RX in the SAME detection row
# ============================================================


def compile_program(rows):

    if not rows:
        raise ValueError("Program contains no rows.")

    # ========================================================
    # 1. VALIDATE USER ROWS
    # ========================================================

    for idx, row in enumerate(rows):

        if "gap_s" in row:
            raise ValueError(
                f"row {idx}: obsolete key 'gap_s' found. "
                "Use 'gap_after_s'."
            )

        if "gap_after_s" not in row:
            raise ValueError(
                f"row {idx}: missing 'gap_after_s'."
            )

        gap = float(row["gap_after_s"])

        if gap < 0:
            raise ValueError(
                f"row {idx}: gap_after_s must be >= 0."
            )

        for lane in row.get("tx", {}):
            if lane not in TX_LANES:
                raise ValueError(
                    f"row {idx}: unknown TX lane {lane!r}"
                )

        for lane in row.get("rx", {}):
            if lane not in RX_LANES:
                raise ValueError(
                    f"row {idx}: unknown RX lane {lane!r}"
                )

        for lane in row.get("refill", []):
            if lane not in TX_LANES:
                raise ValueError(
                    f"row {idx}: refill references "
                    f"unknown TX lane {lane!r}"
                )

    # ========================================================
    # 2. USER gap_after_s -> HARDWARE PRE-ROW GAP
    # ========================================================

    hw_gap_s = []

    for idx in range(len(rows)):

        if idx == 0:
            hw_gap_s.append(
                float(INITIAL_GAP_S)
            )

        else:
            hw_gap_s.append(
                float(
                    rows[idx - 1]["gap_after_s"]
                )
            )

    # ========================================================
    # 3. COMPILER PRODUCTS
    # ========================================================

    table = []

    tx_chunks = {
        lane: []
        for lane in TX_LANES
    }

    tx_chunk_bounds = {
        lane: [0]
        for lane in TX_LANES
    }

    rx_total_s = {
        lane: 0.0
        for lane in RX_LANES
    }

    # Each item:
    #
    #   (
    #       user_row_idx,
    #       lane,
    #       next_chunk_idx,
    #   )
    #
    refill_points = []

    lane_chunk_counter = {
        lane: 1
        for lane in TX_LANES
    }

    # ========================================================
    # 4. COMPILE USER ROWS
    # ========================================================

    for idx, row in enumerate(rows):

        mask = 0
        dur = {}

        # ----------------------------------------------------
        # TX
        # ----------------------------------------------------

        for lane in TX_LANES:

            arr = row.get(
                "tx",
                {}
            ).get(lane)

            if arr is None:
                continue

            if len(arr) == 0:
                raise ValueError(
                    f"row {idx}: {lane} has "
                    "an empty TX waveform."
                )

            mask |= (
                1 << LANE_BIT[lane]
            )

            dur_s = (
                len(arr)
                / SAMPLES_PER_BEAT
                / AXIS_BEAT_HZ
            )

            dur[lane] = beats_for(
                dur_s
            )

            tx_chunks[lane].append(
                arr
            )

        # ----------------------------------------------------
        # RX
        # ----------------------------------------------------

        for lane in RX_LANES:

            dur_s = row.get(
                "rx",
                {}
            ).get(lane)

            if dur_s is None:
                continue

            dur_s = float(dur_s)

            if dur_s <= 0:
                raise ValueError(
                    f"row {idx}: {lane} RX "
                    "duration must be > 0."
                )

            mask |= (
                1 << LANE_BIT[lane]
            )

            dur[lane] = beats_for(
                dur_s
            )

            rx_total_s[lane] += (
                dur_s
            )

        # ----------------------------------------------------
        # REFILL MARKERS
        #
        # Marker on USER row R means:
        #
        #     row R fires
        #         |
        #         | gap_after_s[R]
        #         | software refill here
        #         v
        #     later consuming row fires
        #
        # Chunk boundary is the TX data accumulated so far.
        # ----------------------------------------------------

        for lane in row.get(
            "refill",
            []
        ):

            refill_points.append(
                (
                    idx,
                    lane,
                    lane_chunk_counter[lane],
                )
            )

            lane_chunk_counter[lane] += 1

            tx_chunk_bounds[lane].append(
                sum(
                    len(a)
                    for a in tx_chunks[lane]
                )
            )

        # ----------------------------------------------------
        # HARDWARE ROW
        # ----------------------------------------------------

        table.append(
            {
                "mask": mask,

                "gap_cycles":
                    seq_cycles_for(
                        hw_gap_s[idx]
                    ),

                "dur": dur,
            }
        )

    # ========================================================
    # 5. FLATTEN PER-LANE TX STREAMS
    # ========================================================

    tx_full = {}

    for lane in TX_LANES:

        chunks = tx_chunks[lane]

        if chunks:

            tx_full[lane] = (
                np.concatenate(chunks)
            )

        else:

            tx_full[lane] = (
                np.zeros(
                    0,
                    dtype=np.complex128,
                )
            )

        tx_chunk_bounds[lane].append(
            len(tx_full[lane])
        )

    # ========================================================
    # 6. EXACT HARDWARE ROW FIRE TIMES
    #
    # Use the quantized SEQ_GAP values that will actually be
    # programmed, rather than ideal floating-point gap values.
    # ========================================================

    row_fire_s = []

    t = 0.0

    for hw_row in table:

        t += (
            hw_row["gap_cycles"]
            / SEQ_CLK_HZ
        )

        row_fire_s.append(t)

    # ========================================================
    # 7. REFILL DEADLINES
    #
    # A marker on row R opens its software refill window when
    # row R fires.
    #
    # The hard deadline is the next later row that actually
    # consumes that TX lane.
    # ========================================================

    refill_deadlines = []

    for (
        row_idx,
        lane,
        chunk_idx,
    ) in refill_points:

        start_try_s = (
            row_fire_s[row_idx]
        )

        deadline_s = None
        consume_row = None

        for j in range(
            row_idx + 1,
            len(rows),
        ):

            if lane in rows[j].get(
                "tx",
                {}
            ):

                deadline_s = (
                    row_fire_s[j]
                )

                consume_row = j

                break

        if deadline_s is None:

            raise RuntimeError(
                f"row {row_idx}: refill for "
                f"{lane} has no later TX row "
                "that consumes the new chunk."
            )

        if deadline_s <= start_try_s:

            raise RuntimeError(
                f"row {row_idx}: invalid "
                f"refill window for {lane}."
            )

        refill_deadlines.append(
            (
                row_idx,
                lane,
                chunk_idx,
                start_try_s,
                deadline_s,
                consume_row,
            )
        )

    return (
        table,
        tx_full,
        tx_chunk_bounds,
        refill_deadlines,
        rx_total_s,
        row_fire_s,
        hw_gap_s,
    )


# ============================================================
# 8. COMPILE
# ============================================================

(
    table,
    tx_full,
    tx_chunk_bounds,
    refill_deadlines,
    rx_total_s,
    row_fire_s,
    hw_gap_s,
) = compile_program(rows)


# ============================================================
# 9. BASIC CAPACITY CHECKS
# ============================================================

if len(table) > 256:

    raise ValueError(
        f"Program has {len(table)} rows, "
        "exceeds the 256-row Pulse_Sequencer build."
    )


for lane in TX_LANES:

    print(
        f"{lane}: "
        f"{len(tx_full[lane]):,} "
        "complex samples total across program"
    )


for lane, secs in rx_total_s.items():

    if secs > CAP_MAX_S:

        raise RuntimeError(
            f"{lane}: total capture "
            f"{secs*1e3:.3f} ms exceeds "
            f"{CAP_MAX_S*1e3:.0f} ms "
            "RX DMA limit."
        )

    print(
        f"{lane}: "
        f"{secs*1e3:.3f} ms "
        "total capture this program"
    )


# ============================================================
# 10. PRINT GAP TRANSLATION
# ============================================================

print()
print(
    "=== USER gap_after_s -> "
    "HARDWARE pre-row GAP ==="
)


for i in range(len(rows)):

    if i == 0:

        source = "INITIAL_GAP_S"

    else:

        source = (
            f"user row {i-1} "
            "gap_after_s"
        )

    actual_gap_s = (
        table[i]["gap_cycles"]
        / SEQ_CLK_HZ
    )

    print(
        f"HW row {i:3d}: "
        f"pre-gap="
        f"{actual_gap_s*1e3:10.6f} ms "
        f"<- {source}"
    )


# ============================================================
# 11. BUILD TX EVENT LIST + PRINT COMPILED TIMELINE
# ============================================================

tx_events = []


print()
print(
    "=== COMPILED ROW FIRE TIMELINE ==="
)


for i, row in enumerate(rows):

    parts = []

    # TX
    for lane in TX_LANES:

        arr = row.get(
            "tx",
            {}
        ).get(lane)

        if arr is None:
            continue

        dur_s = (
            len(arr)
            / SAMPLES_PER_BEAT
            / AXIS_BEAT_HZ
        )

        event = {
            "row": i,
            "lane": lane,
            "start": row_fire_s[i],
            "end":
                row_fire_s[i]
                + dur_s,
        }

        tx_events.append(event)

        parts.append(
            f"{lane} "
            f"{dur_s*1e3:.3f}ms "
            f"(ends "
            f"{event['end']*1e3:.3f}ms)"
        )

    # RX
    for lane in RX_LANES:

        dur_s = row.get(
            "rx",
            {}
        ).get(lane)

        if dur_s is None:
            continue

        parts.append(
            f"{lane} "
            f"{float(dur_s)*1e3:.3f}ms"
        )

    print(
        f"row {i:3d} | "
        f"fire="
        f"{row_fire_s[i]*1e3:10.3f} ms | "
        + (
            ", ".join(parts)
            if parts
            else "idle"
        )
    )


# ============================================================
# 12. VERIFY NO UNINTENDED TX OVERLAP BETWEEN DIFFERENT ROWS
#
# IMPORTANT:
#
# Multiple TX lanes in ONE row are intentionally simultaneous.
#
# Example:
#
#   detection:
#       TX_A + TX_B + RX_A + RX_B
#
# is completely legal.
#
# We only reject overlap between DIFFERENT user rows.
# ============================================================

tx_events.sort(
    key=lambda e: (
        e["start"],
        e["row"],
        e["lane"],
    )
)


for i, a in enumerate(tx_events):

    for b in tx_events[i + 1:]:

        # No later event can overlap A after this.
        if (
            b["start"]
            >= a["end"] - 1e-9
        ):
            break

        # Same user row = intentional simultaneous TX.
        if b["row"] == a["row"]:
            continue

        raise RuntimeError(
            "UNINTENDED TX OVERLAP BETWEEN "
            "DIFFERENT ROWS:\n"
            f"  row {a['row']} "
            f"{a['lane']} starts "
            f"{a['start']*1e3:.6f} ms, "
            f"ends "
            f"{a['end']*1e3:.6f} ms\n"
            f"  row {b['row']} "
            f"{b['lane']} starts "
            f"{b['start']*1e3:.6f} ms, "
            f"ends "
            f"{b['end']*1e3:.6f} ms"
        )


print()
print(
    "PASS: no unintended TX overlap "
    "between different rows; simultaneous "
    "TX lanes within one row are allowed."
)


# ============================================================
# 13. FIND FIRST RX ROW
# ============================================================

first_rx_row = None


for i, row in enumerate(rows):

    if row.get("rx", {}):

        first_rx_row = i
        break


if first_rx_row is None:

    raise RuntimeError(
        "Program contains no RX row."
    )


first_rx_fire_s = (
    row_fire_s[first_rx_row]
)


# ============================================================
# 14. VERIFY PRE-DETECTION TX HAS FINISHED
#
# THIS IS THE IMPORTANT FIX.
#
# We deliberately ignore TX events IN the first RX row itself.
#
# A TX event in the same row as RX is intentional:
#
#       row 41:
#           TX_A probe
#           TX_B probe
#           RX_A capture
#           RX_B capture
#
# Those are SUPPOSED to overlap.
#
# We only ask:
#
#     Did every TX event from EARLIER rows finish before
#     the first RX/detection row fired?
# ============================================================

pre_rx_tx_events = [
    event
    for event in tx_events
    if event["row"] < first_rx_row
]


if pre_rx_tx_events:

    last_pre_rx_tx_end_s = max(
        event["end"]
        for event
        in pre_rx_tx_events
    )

    last_pre_rx_event = max(
        pre_rx_tx_events,
        key=lambda event:
            event["end"]
    )

else:

    last_pre_rx_tx_end_s = 0.0
    last_pre_rx_event = None


pre_rx_clearance_s = (
    first_rx_fire_s
    - last_pre_rx_tx_end_s
)


print()
print(
    "=== PRE-DETECTION TX CLEARANCE ==="
)


if last_pre_rx_event is None:

    print(
        "No TX events occur before first RX."
    )

else:

    print(
        f"Last pre-RX TX: "
        f"row {last_pre_rx_event['row']} "
        f"{last_pre_rx_event['lane']}"
    )

    print(
        f"Last pre-RX TX ends: "
        f"{last_pre_rx_tx_end_s*1e3:.3f} ms"
    )


print(
    f"First RX row fires: "
    f"{first_rx_fire_s*1e3:.3f} ms"
)

print(
    f"Pre-RX TX clearance: "
    f"{pre_rx_clearance_s*1e3:+.3f} ms"
)


if pre_rx_clearance_s < -1e-9:

    raise RuntimeError(
        "First RX row fires while a TX event "
        "from an EARLIER row is still active."
    )


print(
    "PASS: all TX events from earlier rows "
    "finish before first RX/detection row."
)


# ============================================================
# 15. REPORT INTENTIONAL TX/RX COINCIDENCE IN FIRST RX ROW
# ============================================================

same_row_tx = []


for lane in TX_LANES:

    arr = rows[
        first_rx_row
    ].get(
        "tx",
        {}
    ).get(lane)

    if arr is not None:

        same_row_tx.append(lane)


print()


if same_row_tx:

    print(
        "First RX row intentionally launches "
        "TX simultaneously: "
        + ", ".join(same_row_tx)
    )

else:

    print(
        "First RX row contains no simultaneous TX."
    )


# ============================================================
# 16. REFILL SCHEDULE
# ============================================================

print()
print(
    "=== REFILL SCHEDULE ==="
)


for (
    row_idx,
    lane,
    chunk_idx,
    start_try_s,
    deadline_s,
    consume_row,
) in refill_deadlines:

    window_s = (
        deadline_s
        - start_try_s
    )

    print(
        f"  after row {row_idx:>3}: "
        f"load {lane} chunk {chunk_idx}; "
        f"consume row={consume_row}, "
        f"try_from="
        f"{start_try_s*1e3:.4f} ms, "
        f"deadline="
        f"{deadline_s*1e3:.4f} ms, "
        f"available="
        f"{window_s*1e3:.4f} ms"
    )


# ============================================================
# 17. DMA CHUNK SIZE CHECKS
# ============================================================

for lane in TX_LANES:

    bounds = (
        tx_chunk_bounds[lane]
    )

    for c0, c1 in zip(
        bounds[:-1],
        bounds[1:],
    ):

        chunk_bytes = (
            (c1 - c0) * 4
        )

        if chunk_bytes > DMA_MAX_BYTES:

            raise RuntimeError(
                f"{lane}: chunk between "
                f"samples {c0}-{c1} is "
                f"{chunk_bytes:,} bytes; "
                f"exceeds DMA maximum "
                f"{DMA_MAX_BYTES:,} bytes."
            )

    chunk_sizes = [
        (b1 - b0) * 4
        for b0, b1
        in zip(
            bounds[:-1],
            bounds[1:],
        )
    ]

    print(
        f"{lane}: "
        f"{len(bounds)-1} chunk(s), "
        f"sizes(bytes)="
        f"{chunk_sizes}"
    )


# ============================================================
# 18. PRINT FINAL HARDWARE ROWS
# ============================================================

print()
print(
    "Final compiled rows:"
)


for i in range(
    max(
        0,
        len(table) - 3,
    ),
    len(table),
):

    r = table[i]

    print(
        f"  row {i}: "
        f"mask=0x{r['mask']:x} "
        f"pre_gap="
        f"{r['gap_cycles']/SEQ_CLK_HZ*1e3:.4f} ms "
        f"DUR0_TXB="
        f"{r['dur'].get('TX_B',0)} "
        f"DUR1_TXA="
        f"{r['dur'].get('TX_A',0)} "
        f"DUR2_RXB="
        f"{r['dur'].get('RX_B',0)} "
        f"DUR3_RXA="
        f"{r['dur'].get('RX_A',0)}"
    )


# ============================================================
# 19. VALIDATE DETECTION ROW
#
# This validation is intentionally GENERAL:
#
# - RX_A and RX_B must match their requested durations.
# - Any TX lane present in the USER detection row must match
#   its waveform-derived duration.
# - A TX lane absent from the USER row must also be absent
#   from the compiled hardware row.
#
# Therefore this works for:
#
#   RX only
#   TX_A + RX
#   TX_B + RX
#   TX_A + TX_B + RX
#
# without rewriting Cell 8 every time.
# ============================================================

_det_user = rows[
    first_rx_row
]

_det_hw = table[
    first_rx_row
]


# Reconstruct expected mask directly from USER row.
expected_mask = 0


for lane in TX_LANES:

    if lane in _det_user.get(
        "tx",
        {}
    ):

        expected_mask |= (
            1 << LANE_BIT[lane]
        )


for lane in RX_LANES:

    if lane in _det_user.get(
        "rx",
        {}
    ):

        expected_mask |= (
            1 << LANE_BIT[lane]
        )


if (
    _det_hw["mask"]
    != expected_mask
):

    raise RuntimeError(
        "Detection row compiled mask "
        f"0x{_det_hw['mask']:x}; "
        f"expected 0x{expected_mask:x}."
    )


# Validate TX durations.
for lane in TX_LANES:

    arr = _det_user.get(
        "tx",
        {}
    ).get(lane)

    actual = (
        _det_hw["dur"].get(
            lane,
            0,
        )
    )

    if arr is None:

        expected = 0

    else:

        dur_s = (
            len(arr)
            / SAMPLES_PER_BEAT
            / AXIS_BEAT_HZ
        )

        expected = beats_for(
            dur_s
        )

    if actual != expected:

        raise RuntimeError(
            f"Detection {lane}: "
            f"compiled duration={actual}, "
            f"expected={expected}."
        )


# Validate RX durations.
for lane in RX_LANES:

    requested = (
        _det_user.get(
            "rx",
            {}
        ).get(lane)
    )

    actual = (
        _det_hw["dur"].get(
            lane,
            0,
        )
    )

    if requested is None:

        expected = 0

    else:

        expected = beats_for(
            float(requested)
        )

    if actual != expected:

        raise RuntimeError(
            f"Detection {lane}: "
            f"compiled duration={actual}, "
            f"expected={expected}."
        )


print()
print(
    "PASS: first RX/detection row compiled "
    "exactly from its user TX/RX definition."
)


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print()
print(
    "PASS: program compiled with natural "
    "gap_after_s semantics."
)



TX_A: 4,921,840 complex samples total across program
TX_B: 49,158,640 complex samples total across program
RX_A: 0.059 ms total capture this program
RX_B: 0.059 ms total capture this program

=== USER gap_after_s -> HARDWARE pre-row GAP ===
HW row   0: pre-gap=  0.000080 ms <- INITIAL_GAP_S
HW row   1: pre-gap=  2.005000 ms <- user row 0 gap_after_s
HW row   2: pre-gap= 20.005003 ms <- user row 1 gap_after_s
HW row   3: pre-gap=  2.005000 ms <- user row 2 gap_after_s
HW row   4: pre-gap= 20.005003 ms <- user row 3 gap_after_s
HW row   5: pre-gap=  2.005000 ms <- user row 4 gap_after_s
HW row   6: pre-gap= 20.005003 ms <- user row 5 gap_after_s
HW row   7: pre-gap=  2.005000 ms <- user row 6 gap_after_s
HW row   8: pre-gap= 20.005003 ms <- user row 7 gap_after_s
HW row   9: pre-gap=  2.005000 ms <- user row 8 gap_after_s
HW row  10: pre-gap= 20.005003 ms <- user row 9 gap_after_s
HW row  11: pre-gap=  2.005000 ms <- user row 10 gap_after_s
HW row  12: pre-gap= 20.005003 ms <- user row 1

## Cell 10 -- RFDC NCO setup + DMA buffers (one buffer per TX lane, sliced for chunks)

The fix for "why 20 buffers": each lane's whole waveform is quantized once into a single PL
buffer; refill chunks are slices of that one buffer, not separate allocations. RX is a single
buffer per lane (no double buffering -- nothing to overlap without Ethernet).

In [9]:
# ================= RFDC NCO SETUP =================
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    ms['Freq'] = float(freq)
    ms['PhaseOffset'] = 0.0
    ms['EventSource'] = 2
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    assert abs(float(dac.MixerSettings['Freq']) - freq) < 1e-6
print('PASS: DAC mixer state matches config.')

# ================= DMA BUFFERS (single allocation per lane) =================
tx_full_buf = {}     # lane -> one PynqBuffer holding the WHOLE quantized waveform
tx_chunk_view = {}    # lane -> list of slices (views) into tx_full_buf[lane], one per chunk
for lane in TX_LANES:
    quantized = quantize_iq(tx_full[lane])           # one quantize_iq() call for the whole lane
    buf = base.device.get_memory_by_idx(1).allocate(shape=quantized.shape, dtype=np.int16)
    buf[:] = quantized
    buf.flush()                                       # one flush for the whole buffer
    tx_full_buf[lane] = buf

    bounds = tx_chunk_bounds[lane]
    views = [buf[2*c0:2*c1] for c0, c1 in zip(bounds[:-1], bounds[1:])]
    tx_chunk_view[lane] = views

    # Sanity-check the key assumption this whole design rests on: that a slice of a PynqBuffer
    # reports the correct physical address (base + byte offset), so transfer()ing a slice
    # actually points the DMA at the right bytes instead of silently re-sending chunk 0.
    base_pa = int(buf.physical_address)
    for k, (c0, c1) in enumerate(zip(bounds[:-1], bounds[1:])):
        expected_pa = base_pa + 2*c0*2   # 2 int16/sample * 2 bytes/int16
        actual_pa = int(views[k].physical_address)
        if actual_pa != expected_pa:
            raise RuntimeError(f"{lane} chunk {k}: slice physical_address 0x{actual_pa:x} != "
                                f"expected 0x{expected_pa:x}. Buffer slicing does not behave as "
                                f"assumed on this PYNQ version -- do not trust sliced transfers; "
                                f"fall back to one allocate() per chunk instead.")
    print(f"{lane}: 1 buffer allocated ({buf.nbytes:,} bytes), {len(views)} chunk view(s), "
          f"physical addresses verified.")

rx_buffers = {}
for lane in RX_LANES:
    capture_beats = beats_for(rx_total_s[lane])
    capture_samples = samples_for(capture_beats)
    rx_buffers[lane] = allocate(shape=(capture_samples,), dtype=np.int16)
    print(f"{lane}: RX buffer {rx_buffers[lane].nbytes:,} bytes (single, reused every shot).")

print('PASS: DMA buffers allocated.')


PASS: DAC mixer state matches config.
TX_A: 1 buffer allocated (19,687,360 bytes), 1 chunk view(s), physical addresses verified.
TX_B: 1 buffer allocated (196,634,560 bytes), 20 chunk view(s), physical addresses verified.
RX_A: RX buffer 14,512 bytes (single, reused every shot).
RX_B: RX buffer 14,512 bytes (single, reused every shot).
PASS: DMA buffers allocated.


## Cell 11 -- Commit the compiled table to the sequencer

In [10]:
# ================= COMMIT TABLE TO HARDWARE =================
def commit_table(table):
    seq.mmio.write(SEQ_TTL_MODE, 0)
    seq.mmio.write(SEQ_ENABLE, 0)
    seq.mmio.write(SEQ_RESET, 1)
    time.sleep(100e-6)
    for idx, row in enumerate(table):
        seq.mmio.write(SEQ_ROW_SEL, idx)
        seq.mmio.write(SEQ_MASK, row["mask"])
        seq.mmio.write(SEQ_GAP, row["gap_cycles"])
        seq.mmio.write(SEQ_DUR0, row["dur"].get("TX_B", 0))
        seq.mmio.write(SEQ_DUR1, row["dur"].get("TX_A", 0))
        seq.mmio.write(SEQ_DUR2, row["dur"].get("RX_B", 0))
        seq.mmio.write(SEQ_DUR3, row["dur"].get("RX_A", 0))
        seq.mmio.write(SEQ_COMMIT, 1)
    seq.mmio.write(SEQ_TABLE_LEN, len(table))
    print(f"PASS: committed {len(table)} rows to the sequencer.")

commit_table(table)


PASS: committed 42 rows to the sequencer.


## Cell 12 -- DMA/loop helpers + `dump_state()` (no Ethernet, no threading)

In [11]:
# ================= DMA / LOOP HELPERS =================
def dma_status(ch): return int(ch._mmio.read(int(ch._offset)+0x04))
def dma_flags(v):
    out = ['halted' if v&1 else 'running']
    if v&2: out.append('idle')
    if v&0x10: out.append('INTERNAL_ERR')
    if v&0x20: out.append('SLAVE_ERR')
    if v&0x40: out.append('DECODE_ERR')
    return '|'.join(out)

def ensure_running(ch, label):
    st = dma_status(ch)

    if st & DMASR_ERR_MASK:
        raise RuntimeError(
            f"{label}: DMA error before shot: "
            f"0x{st:08x} ({dma_flags(st)})"
        )

    if st & DMASR_HALTED:
        safe_start(ch, label, timeout_s=0.2)

        st = dma_status(ch)

        if st & DMASR_ERR_MASK:
            raise RuntimeError(
                f"{label}: DMA error after restart: "
                f"0x{st:08x} ({dma_flags(st)})"
            )

        if st & DMASR_HALTED:
            raise RuntimeError(
                f"{label}: still HALTED after restart: "
                f"0x{st:08x} ({dma_flags(st)})"
            )

def wait_dma_idle(ch, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = dma_status(ch)
        if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error: 0x{st:08x} ({dma_flags(st)})')
        if st & DMASR_IDLE: return
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: DMA not idle: 0x{st:08x} ({dma_flags(st)})')
        time.sleep(50e-6)

def wait_cap_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(CAP_STATUS))
        if not (st & CAP_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: Capture_Gate busy: 0x{st:08x}')
        time.sleep(100e-6)

def wait_tx_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(TXM_STATUS))
        if not (st & TX_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: TX gate busy: 0x{st:08x}')
        time.sleep(20e-6)

def discard_rx(buf):
    inv = getattr(buf, 'invalidate', None)
    if inv:
        try: inv()
        except Exception: pass

def ttl_arm(s): s.mmio.write(SEQ_TTL_ARM, 1)

def dump_state(label=""):
    print(f"---- dump_state: {label} ----")
    try:
        print(f"  seq:   ROW_PTR={int(seq.mmio.read(SEQ_ROW_PTR))} "
              f"TTL_STATUS=0x{int(seq.mmio.read(SEQ_TTL_STATUS)):x} "
              f"CDC_OVERRUN=0x{int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF:x}")
    except Exception as e:
        print(f"  seq: <read failed: {e}>")
    for name, g in [("tx_a", tx_a), ("tx_b", tx_b)]:
        try:
            st = int(g.mmio.read(TXM_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&TX_BUSY)}, overrun={bool(st&TX_OVERRUN)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, g in [("cap_a", cap_a), ("cap_b", cap_b)]:
        try:
            st = int(g.mmio.read(CAP_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&CAP_BUSY)}, overflow={bool(st&CAP_OVERFLOW)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, ch in [("RX-A", dma_ra.recvchannel), ("RX-B", dma_rb.recvchannel),
                      ("TX-A", dma_ta.sendchannel), ("TX-B", dma_tb.sendchannel)]:
        try:
            st = dma_status(ch)
            print(f"  {name}: 0x{st:08x} ({dma_flags(st)}) transferred={int(ch.transferred)}")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    print("-" * (18 + len(label)))

print('DMA/loop helpers ready.')


DMA/loop helpers ready.


## Ethernet helpers -- same synchronous protocol as the known-working qualification notebook


In [12]:
# ================= ETHERNET HELPERS -- mirrors known-working qualification notebook =================
import socket, json, struct

MAGIC_HELLO = b'RFDC'
MAGIC_SHOT  = b'SHOT'
MAGIC_DROP  = b'DROP'
ACK = b'OK'

def _recv_exact(sock, n):
    buf = bytearray(n)
    view = memoryview(buf)
    got = 0
    while got < n:
        r = sock.recv_into(view[got:], n - got)
        if r == 0:
            raise ConnectionError('PC closed the connection mid-transfer')
        got += r
    return bytes(buf)

def eth_connect(host, port, timeout_s):
    s = socket.create_connection((host, port), timeout=timeout_s)
    s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    return s

def _send_json(sock, magic, obj):
    payload = json.dumps(obj).encode('utf-8')
    sock.sendall(magic + struct.pack('>I', len(payload)) + payload)

def eth_handshake(sock, meta):
    _send_json(sock, MAGIC_HELLO, meta)
    ack = _recv_exact(sock, 2)
    if ack != ACK:
        raise RuntimeError(f'PC receiver did not ACK handshake, got {ack!r}')

def eth_send_shot(sock, shot_num, header, rx_a, rx_b):
    hdr = dict(header)
    hdr['shot'] = int(shot_num)
    _send_json(sock, MAGIC_SHOT, hdr)
    sock.sendall(memoryview(np.asarray(rx_a)))
    sock.sendall(memoryview(np.asarray(rx_b)))
    ack = _recv_exact(sock, 2)
    if ack != ACK:
        raise RuntimeError(f'shot {shot_num}: PC receiver did not ACK, got {ack!r}')

def eth_send_drop(sock, shot_num, header):
    hdr = dict(header); hdr['shot'] = int(shot_num)
    _send_json(sock, MAGIC_DROP, hdr)
    ack = _recv_exact(sock, 2)
    if ack != ACK:
        raise RuntimeError(f'drop {shot_num}: PC receiver did not ACK, got {ack!r}')

print('Ethernet helpers ready (SHOT/raw/raw/ACK plus DROP/ACK).')


Ethernet helpers ready (SHOT/raw/raw/ACK plus DROP/ACK).


## Connect to PC + handshake

Run the PC receiver first. This notebook does not start the hardware loop until the receiver ACKs the metadata handshake.


In [13]:
# ================= CONNECT TO PC + HANDSHAKE =================
# Start the PC receiver notebook first, then run this cell.
rx_capture_samples = {lane: int(len(rx_buffers[lane])) for lane in RX_LANES}
rx_capture_bytes = {lane: int(rx_buffers[lane].nbytes) for lane in RX_LANES}

handshake_meta = dict(
    protocol='RFDC_ROWPROGRAM_OVERLAP_V1',
    AXIS_BEAT_HZ=AXIS_BEAT_HZ,
    SAMPLES_PER_BEAT=SAMPLES_PER_BEAT,
    SEQ_CLK_HZ=SEQ_CLK_HZ,
    fs_hz=FS_HZ,
    DAC_A_NCO_MHZ=DAC_A_NCO_MHZ,
    DAC_B_NCO_MHZ=DAC_B_NCO_MHZ,
    rx_lanes=list(RX_LANES),
    rx_total_s={k: float(v) for k,v in rx_total_s.items()},
    rx_capture_samples=rx_capture_samples,
    rx_capture_bytes=rx_capture_bytes,
    row_fire_s=[float(x) for x in row_fire_s],
    table_rows=len(table),
    LOOP_COUNT=int(LOOP_COUNT),
)

print(f'Connecting to PC receiver at {PC_HOST}:{PC_PORT} ...')
eth_sock = eth_connect(PC_HOST, PC_PORT, ETH_TIMEOUT_S)
print('Connected. Sending row-program acquisition metadata...')
eth_handshake(eth_sock, handshake_meta)
print('PASS: PC receiver ACKed handshake; overlap streaming can begin.')


Connecting to PC receiver at 192.168.3.139:5001 ...
Connected. Sending row-program acquisition metadata...
PASS: PC receiver ACKed handshake; overlap streaming can begin.


## Cell 13 -- Run loop: previous Ethernet overlaps next shot until RX-arm deadline


In [14]:
# ================= RUN LOOP: ETHERNET OVERLAPS PRE-CAPTURE ROWS =================
import threading

# Find the first row that fires either RX lane. This is the hard ownership deadline.
RX_ROWS = [i for i,row in enumerate(rows) if any(l in row.get("rx", {}) for l in RX_LANES)]
if not RX_ROWS:
    raise RuntimeError("Program has no RX rows; overlap test expects at least one capture row.")
FIRST_RX_ROW = RX_ROWS[0]
FIRST_RX_FIRE_S = float(row_fire_s[FIRST_RX_ROW])
RX_ARM_DEADLINE_S = max(0.0, FIRST_RX_FIRE_S - RX_ARM_GUARD_S)
RX_PAYLOAD_BYTES = int(rx_buffers["RX_A"].nbytes + rx_buffers["RX_B"].nbytes)
ETH_MODEL_S = RX_PAYLOAD_BYTES * 8.0 / (ETHERNET_MODEL_MBPS * 1e6) + ETHERNET_MODEL_MARGIN_S
PREDICTED_SLACK_S = RX_ARM_DEADLINE_S - ETH_MODEL_S
print(f"First RX row={FIRST_RX_ROW}, fires at {FIRST_RX_FIRE_S*1e3:.3f} ms")
print(f"RX arm deadline={RX_ARM_DEADLINE_S*1e3:.3f} ms (guard {RX_ARM_GUARD_S*1e3:.1f} ms)")
print(f"RX payload={RX_PAYLOAD_BYTES/1e6:.3f} MB; conservative {ETHERNET_MODEL_MBPS:.0f} Mbps model + {ETHERNET_MODEL_MARGIN_S*1e3:.1f} ms margin => {ETH_MODEL_S*1e3:.1f} ms")
print(f"Predicted overlap slack={PREDICTED_SLACK_S*1e3:+.1f} ms (diagnostic only; runtime uses actual completion state)")

class EthernetJob:
    def __init__(self):
        self.thread = None; self.done = threading.Event(); self.done.set()
        self.exc = None; self.elapsed_s = None; self.shot = None; self.kind = None
        self.start_t = None; self.end_t = None
    def busy(self): return self.thread is not None and self.thread.is_alive()
    def check(self):
        if self.exc is not None:
            e=self.exc; self.exc=None; raise RuntimeError(f"Ethernet worker failed: {e}") from e
    def wait(self, timeout=None):
        ok=self.done.wait(timeout)
        if ok: self.check()
        return ok
    def _launch(self, kind, shot, fn):
        if self.busy(): raise RuntimeError("attempted to launch Ethernet while previous job still busy")
        self.done.clear(); self.exc=None; self.elapsed_s=None; self.shot=shot; self.kind=kind
        def worker():
            self.start_t=time.perf_counter()
            try: fn()
            except BaseException as e: self.exc=e
            finally:
                self.end_t=time.perf_counter(); self.elapsed_s=self.end_t-self.start_t; self.done.set()
        self.thread=threading.Thread(target=worker, name=f"eth-{kind}-{shot}", daemon=True); self.thread.start()
    def launch_shot(self, shot, header):
        self._launch('SHOT', shot, lambda: eth_send_shot(eth_sock, shot, header, rx_buffers['RX_A'], rx_buffers['RX_B']))
    def launch_drop(self, shot, header):
        self._launch('DROP', shot, lambda: eth_send_drop(eth_sock, shot, header))

eth_job = EthernetJob()


def prepare_common():
    wait_tx_idle(tx_a, "TX_A", 1.0); wait_tx_idle(tx_b, "TX_B", 1.0)
    tx_a.mmio.write(TXM_STATUS, TX_OVERRUN); tx_b.mmio.write(TXM_STATUS, TX_OVERRUN)
    seq.mmio.write(SEQ_ENABLE, 0); seq.mmio.write(SEQ_RESET, 1); time.sleep(100e-6)
    seq.mmio.write(SEQ_CDC_OVERRUN, 0xF); seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1); seq.mmio.write(SEQ_TTL_MODE, 1)
    for ch,label in [(dma_ta.sendchannel,'TX-A'),(dma_tb.sendchannel,'TX-B')]: ensure_running(ch,label)


def prepare_rx_for_capture():
    wait_cap_idle(cap_a, "RX_A", 2.0); wait_cap_idle(cap_b, "RX_B", 2.0)
    cap_a.mmio.write(CAP_CLEAR, CAP_OVERFLOW); cap_b.mmio.write(CAP_CLEAR, CAP_OVERFLOW)
    cap_a.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_A"])); cap_b.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_B"]))
    for ch,label in [(dma_ra.recvchannel,'RX-A'),(dma_rb.recvchannel,'RX-B')]: ensure_running(ch,label)
    dma_ra.recvchannel.transfer(rx_buffers["RX_A"]); dma_rb.recvchannel.transfer(rx_buffers["RX_B"])


def arm_initial_tx():
    dma_ta.sendchannel.transfer(tx_chunk_view["TX_A"][0]); dma_tb.sendchannel.transfer(tx_chunk_view["TX_B"][0])
    time.sleep(5e-3)
    for ch,label in [(dma_ta.sendchannel,'TX-A'),(dma_tb.sendchannel,'TX-B')]:
        st=dma_status(ch)
        if st & (DMASR_ERR_MASK|DMASR_HALTED): raise RuntimeError(f"{label}: bad after arm 0x{st:08x} ({dma_flags(st)})")

DMA_LANE_TO_CHANNEL={"TX_A":dma_ta.sendchannel,"TX_B":dma_tb.sendchannel}
def _try_refill(row_idx,lane,chunk_idx,start_try_s,deadline_s,consume_row,t_run,margins):
    if chunk_idx >= len(tx_chunk_view[lane]): return True
    ch=DMA_LANE_TO_CHANNEL[lane]; st=dma_status(ch)
    if st & DMASR_ERR_MASK: raise RuntimeError(f"{lane}: DMA error before refill 0x{st:08x} ({dma_flags(st)})")
    if not (st & DMASR_IDLE): return False
    t0=time.perf_counter(); ch.transfer(tx_chunk_view[lane][chunk_idx]); t1=time.perf_counter()
    elapsed=t1-t_run; margin=None if deadline_s is None else deadline_s-elapsed
    if margin is not None: margins.append(margin)
    if DEBUG_VERBOSE:
        print(f"  refill {lane} chunk {chunk_idx}: gap-before-row={row_idx}, consume-row={consume_row}, issued={elapsed*1e3:.4f} ms, transfer_call={(t1-t0)*1e3:.4f} ms, margin={margin*1e3:+.4f} ms")
    return True


def run_shot(k, first=False):
    t0=time.perf_counter(); prepare_common(); arm_initial_tx()
    rx_armed=False; rx_dropped=False; drop_reason=None; rx_arm_time_s=None
    # First shot has no previous Ethernet owner, so arm RX before trigger.
    if first:
        prepare_rx_for_capture(); rx_armed=True
    pending=[tuple(x) for x in refill_deadlines]
    ttl_arm(seq); t_run=time.perf_counter(); margins=[]; deadline_wall=t_run+5.0
    tx_diag_pre = None
    tx_diag_post = None
    tx_diag_pre_t = None
    tx_diag_post_t = None
    while True:
        elapsed=time.perf_counter()-t_run

        # Diagnostic snapshots around the first RX firing time. At this point all TX-B
        # refills are already complete, so these AXI-Lite reads do not compete with refill deadlines.
        if tx_diag_pre is None and elapsed >= max(0.0, FIRST_RX_FIRE_S - 1e-3):
            tx_diag_pre_t = elapsed
            tx_diag_pre = (int(tx_a.mmio.read(TXM_STATUS)), int(tx_b.mmio.read(TXM_STATUS)))
        if tx_diag_post is None and elapsed >= FIRST_RX_FIRE_S:
            tx_diag_post_t = elapsed
            tx_diag_post = (int(tx_a.mmio.read(TXM_STATUS)), int(tx_b.mmio.read(TXM_STATUS)))

        # As soon as previous Ethernet releases the single RX buffers, arm them for this run.
        if (not rx_armed) and (not rx_dropped):
            if eth_job.done.is_set():
                eth_job.check()
                if elapsed < RX_ARM_DEADLINE_S:
                    prepare_rx_for_capture(); rx_armed=True; rx_arm_time_s=time.perf_counter()-t_run
                else:
                    rx_dropped=True; drop_reason='ETHERNET_BUSY_AT_RX_ARM_DEADLINE'
            elif elapsed >= RX_ARM_DEADLINE_S:
                rx_dropped=True; drop_reason='ETHERNET_BUSY_AT_RX_ARM_DEADLINE'
                print(f"  shot {k}: RX DROPPED -- previous Ethernet still busy at {elapsed*1e3:.3f} ms; sequencer continues")
        while pending:
            ri,lane,ci,start_s,deadline_s,consume_row=pending[0]
            if elapsed < start_s: break
            if deadline_s is not None and elapsed >= deadline_s: raise RuntimeError(f"shot {k}: {lane} chunk {ci} missed refill deadline")
            if _try_refill(ri,lane,ci,start_s,deadline_s,consume_row,t_run,margins):
                pending.pop(0); elapsed=time.perf_counter()-t_run
            else: break
        cdc=int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF
        if cdc: raise RuntimeError(f"shot {k}: CDC overrun 0x{cdc:x}")
        if int(tx_a.mmio.read(TXM_STATUS)) & TX_OVERRUN: raise RuntimeError(f"shot {k}: TX-A CONTROL_OVERRUN")
        if int(tx_b.mmio.read(TXM_STATUS)) & TX_OVERRUN: raise RuntimeError(f"shot {k}: TX-B CONTROL_OVERRUN")
        ttl_status=int(seq.mmio.read(SEQ_TTL_STATUS))
        if ttl_status & TTL_RUN_DONE: break
        if time.perf_counter()>deadline_wall: raise TimeoutError(f"shot {k}: RUN_DONE timeout")
        time.sleep(100e-6)
    if pending: raise RuntimeError(f"shot {k}: RUN_DONE with {len(pending)} pending refills")
    seq.mmio.write(SEQ_TTL_STATUS_CLEAR,1)
    row=int(seq.mmio.read(SEQ_ROW_PTR))
    # TX always completes, whether RX was captured or dropped.
    wait_dma_idle(dma_ta.sendchannel,'TX-A',2.0); wait_dma_idle(dma_tb.sendchannel,'TX-B',2.0)
    wait_tx_idle(tx_a,'TX-A',2.0); wait_tx_idle(tx_b,'TX-B',2.0)
    overflow=False; rx_a_bytes=0; rx_b_bytes=0
    if rx_armed:
        tout=max(2.0,max(rx_total_s.values())+1.0)
        wait_cap_idle(cap_a,'RX_A',tout); wait_cap_idle(cap_b,'RX_B',tout)
        wait_dma_idle(dma_ra.recvchannel,'RX-A',tout); wait_dma_idle(dma_rb.recvchannel,'RX-B',tout)
        dma_ra.recvchannel.wait(); dma_rb.recvchannel.wait()
        ca=int(cap_a.mmio.read(CAP_STATUS)); cb=int(cap_b.mmio.read(CAP_STATUS)); overflow=bool(ca&CAP_OVERFLOW) or bool(cb&CAP_OVERFLOW)
        discard_rx(rx_buffers['RX_A']); discard_rx(rx_buffers['RX_B'])
        rx_a_bytes=int(dma_ra.recvchannel.transferred); rx_b_bytes=int(dma_rb.recvchannel.transferred)
        if rx_a_bytes != int(rx_buffers['RX_A'].nbytes) or rx_b_bytes != int(rx_buffers['RX_B'].nbytes):
            raise RuntimeError(f"shot {k}: RX byte mismatch {rx_a_bytes}/{rx_b_bytes}")
    else:
        # Capture_Gate is non-stallable; the hardware run is not delayed. We intentionally do
        # not reuse the RX buffers. The next eligible capture clears sticky overflow before arm.
        overflow=True
    dma_ta.sendchannel.wait(); dma_tb.sendchannel.wait()
    final_cdc=int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF
    if final_cdc: raise RuntimeError(f"shot {k}: post-run CDC overrun 0x{final_cdc:x}")
    return dict(shot=k,row=row,captured=bool(rx_armed),dropped=bool(rx_dropped),drop_reason=drop_reason,
                overflow=bool(overflow),rx_a_bytes=rx_a_bytes,rx_b_bytes=rx_b_bytes,
                min_margin_s=(min(margins) if margins else None),rx_arm_time_s=rx_arm_time_s,
                hardware_elapsed_s=time.perf_counter()-t0,
                tx_diag_pre=tx_diag_pre, tx_diag_post=tx_diag_post,
                tx_diag_pre_t=tx_diag_pre_t, tx_diag_post_t=tx_diag_post_t)

print('Overlap run-loop ready.')


First RX row=41, fires at 445.205 ms
RX arm deadline=440.205 ms (guard 5.0 ms)
RX payload=0.029 MB; conservative 200 Mbps model + 10.0 ms margin => 11.2 ms
Predicted overlap slack=+429.0 ms (diagnostic only; runtime uses actual completion state)
Overlap run-loop ready.


## Cell 14 -- Run + overlap/drop summary


In [15]:
# ================= RUN + OVERLAPPED ETHERNET =================
results=[]; previous_eth_elapsed=None
print(f"RUNNING {LOOP_COUNT} shots. Ethernet model={ETHERNET_MODEL_MBPS:.0f} Mbps; Pulse Sequencer is never delayed for Ethernet.")
for k in range(1, LOOP_COUNT+1):
    try:
        r=run_shot(k, first=(k==1)); results.append(r)
        # The socket must be free here if this shot captured; if it dropped, previous Ethernet may
        # still be finishing. A DROP notification is sent once the socket becomes free, but this
        # bookkeeping never changes the just-completed hardware timing.
        if r['captured']:
            if not eth_job.done.is_set():
                raise RuntimeError('captured shot completed while previous Ethernet still owns RX buffers -- ownership bug')
            eth_job.check()
            eth_job.launch_shot(k, {"row":r['row'],"captured":True,"dropped":False,"overflow":r['overflow'],
                                    "cdc":0,"rx_a_bytes":r['rx_a_bytes'],"rx_b_bytes":r['rx_b_bytes'],
                                    "min_refill_margin_s":r['min_margin_s'],"rx_arm_time_s":r['rx_arm_time_s']})
        else:
            # Wait only for protocol serialization AFTER the run; this does not retroactively delay it.
            # For the present overlap bench, a miss is explicitly logged. Future externally-clocked
            # scheduling can queue DROP metadata without waiting here.
            eth_job.wait(ETH_TIMEOUT_S)
            eth_job.launch_drop(k,{"captured":False,"dropped":True,"drop_reason":r['drop_reason'],"overflow":True,
                                   "min_refill_margin_s":r['min_margin_s']})
        mm=r['min_margin_s']; mm_s='n/a' if mm is None else f"{mm*1e3:+.4f} ms"
        arm_s='pre-trigger' if k==1 else ('DROPPED' if r['rx_arm_time_s'] is None else f"{r['rx_arm_time_s']*1e3:.2f} ms")
        print(f"shot {k:04d}: captured={r['captured']} rx_arm={arm_s} min_refill={mm_s} hardware={r['hardware_elapsed_s']:.3f}s eth_prev_busy_now={eth_job.busy()}")
    except Exception:
        print(f"shot {k} FAILED"); dump_state(f"shot {k} exception")
        if FAIL_FAST: raise
# Finish final asynchronous send before closing socket.
eth_job.wait(ETH_TIMEOUT_S)
print(); print(f"Completed {len(results)}/{LOOP_COUNT} scheduled shots.")
drops=[r for r in results if r['dropped']]
print(f"Captured={len(results)-len(drops)}, dropped={len(drops)}")
if drops: print('Dropped shot numbers:', [r['shot'] for r in drops])
all_m=[r['min_margin_s'] for r in results if r['min_margin_s'] is not None]
if all_m: print(f"Worst TX refill margin: {min(all_m)*1e3:+.4f} ms")
print(f"Conservative model: payload={RX_PAYLOAD_BYTES/1e6:.3f} MB, eth_est={ETH_MODEL_S*1e3:.1f} ms, first_RX={FIRST_RX_FIRE_S*1e3:.1f} ms, predicted_slack={PREDICTED_SLACK_S*1e3:+.1f} ms")
seq.mmio.write(SEQ_TTL_MODE,0); seq.mmio.write(SEQ_ENABLE,0)
try: eth_sock.shutdown(socket.SHUT_RDWR)
except Exception: pass
eth_sock.close(); print('Ethernet connection closed. Done.')


RUNNING 5 shots. Ethernet model=200 Mbps; Pulse Sequencer is never delayed for Ethernet.
  refill TX_B chunk 1: gap-before-row=2, consume-row=3, issued=22.2629 ms, transfer_call=0.1630 ms, margin=+1.7522 ms
  refill TX_B chunk 2: gap-before-row=4, consume-row=5, issued=44.2863 ms, transfer_call=0.1368 ms, margin=+1.7388 ms
  refill TX_B chunk 3: gap-before-row=6, consume-row=7, issued=66.3221 ms, transfer_call=0.1344 ms, margin=+1.7130 ms
  refill TX_B chunk 4: gap-before-row=8, consume-row=9, issued=88.3380 ms, transfer_call=0.1326 ms, margin=+1.7071 ms
  refill TX_B chunk 5: gap-before-row=10, consume-row=11, issued=110.3442 ms, transfer_call=0.1327 ms, margin=+1.7109 ms
  refill TX_B chunk 6: gap-before-row=12, consume-row=13, issued=132.4275 ms, transfer_call=0.1344 ms, margin=+1.6376 ms
  refill TX_B chunk 7: gap-before-row=14, consume-row=15, issued=154.3184 ms, transfer_call=0.1340 ms, margin=+1.7567 ms
  refill TX_B chunk 8: gap-before-row=16, consume-row=17, issued=176.4287 ms